## 1. Setup

In [1]:
from pathlib import Path
import json
import os
import tempfile
import time

import faiss
import numpy as np
import pandas as pd

In [ ]:
BASE_DIR = Path.cwd()

EMB_PATH = BASE_DIR / "data/faiss/full_embeddings.npy"
META_PATH = BASE_DIR / "data/faiss/metadata.parquet"

OUT_DIR = BASE_DIR / "outputs/benchmark"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_SIZE = 100_000
N_QUERIES = 200
TOPK = 10
SEED = 42

MIN_RECALL = 0.97

IVF_NLIST = 1024 # số cụm
IVF_TRAIN_SIZE = 50_000 # số lượng vector train
IVF_NPROBES = [4, 8, 16, 32, 64] # search số cụm gần query nhất

HNSW_M = 32 # node kết nối lân cận trong graph
HNSW_EF_CONSTRUCTION = 80 # x mới thêm vào graph -> tìm 80 ứng viên gần x
HNSW_EF_SEARCHES = [16, 32, 64, 128] # mở rộng node tìm kết quả

## 2. Load embedding sample

In [3]:
x_all = np.load(EMB_PATH, mmap_mode="r")

print("embedding shape:", x_all.shape)
print("dtype:", x_all.dtype)

embedding shape: (1347729, 384)
dtype: float32


In [4]:
if META_PATH.exists():
    meta = pd.read_parquet(META_PATH)
    print("metadata rows:", len(meta))
    print("match:", len(meta) == x_all.shape[0])
else:
    meta = None

metadata rows: 1347729
match: True


In [5]:
n_take = min(SAMPLE_SIZE, x_all.shape[0])

xb = np.array(x_all[:n_take], dtype="float32", copy=True)
xb = np.ascontiguousarray(xb)
faiss.normalize_L2(xb)

n, d = xb.shape
n, d

(100000, 384)

## 3. Query sample

In [6]:
rng = np.random.default_rng(SEED)

qids = rng.choice(n, size=min(N_QUERIES, n), replace=False)
xq = xb[qids].copy()
xq = np.ascontiguousarray(xq.astype("float32"))
faiss.normalize_L2(xq)

xq.shape

(200, 384)

## 4. Hàm benchmark

In [7]:
def get_size_mb(index):
    f = tempfile.NamedTemporaryFile(delete=False, suffix=".index")
    path = f.name
    f.close()

    faiss.write_index(index, path)
    size = os.path.getsize(path) / 1024 / 1024
    os.remove(path)
    return round(size, 2)


def recall_at_k(pred, gt, k):
    score = []
    for a, b in zip(pred[:, :k], gt[:, :k]):
        score.append(len(set(a).intersection(set(b))) / k)
    return float(np.mean(score))


def run_search(index, name, gt_ids):
    index.search(xq[:5], TOPK)

    t0 = time.perf_counter()
    _, ids = index.search(xq, TOPK)
    sec = time.perf_counter() - t0

    return {
        "index_name": name,
        "search_sec": round(sec, 4),
        "ms_per_query": round(sec * 1000 / len(xq), 4),
        "qps": round(len(xq) / sec, 2),
        f"recall@{TOPK}": round(recall_at_k(ids, gt_ids, TOPK), 4),
    }

## 5. FlatIP baseline

In [8]:
results = []

t0 = time.perf_counter()
flat = faiss.IndexFlatIP(d)
flat.add(xb)
build_sec = time.perf_counter() - t0

_, gt_ids = flat.search(xq, TOPK)

row = run_search(flat, "FlatIP_exact", gt_ids)
row["build_sec"] = round(build_sec, 4)
row["size_mb"] = get_size_mb(flat)
results.append(row)

row

{'index_name': 'FlatIP_exact',
 'search_sec': 5.6617,
 'ms_per_query': 28.3083,
 'qps': 35.33,
 'recall@10': 1.0,
 'build_sec': 0.2744,
 'size_mb': 146.48}

## 6. IVFFlat

In [9]:
nlist = min(IVF_NLIST, max(16, n // 40))
train_size = min(IVF_TRAIN_SIZE, n)

train_ids = rng.choice(n, size=train_size, replace=False)
xtrain = xb[train_ids].copy()
xtrain = np.ascontiguousarray(xtrain.astype("float32"))
faiss.normalize_L2(xtrain)

quantizer = faiss.IndexFlatIP(d)
ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)

t0 = time.perf_counter()
ivf.train(xtrain)
ivf.add(xb)
build_sec = time.perf_counter() - t0
size_mb = get_size_mb(ivf)

for nprobe in IVF_NPROBES:
    if nprobe > nlist:
        continue

    ivf.nprobe = nprobe
    row = run_search(ivf, f"IVFFlat_nprobe={nprobe}", gt_ids)
    row["build_sec"] = round(build_sec, 4)
    row["size_mb"] = size_mb
    results.append(row)

pd.DataFrame(results)

,index_name,search_sec,ms_per_query,qps,recall@10,build_sec,size_mb
0,FlatIP_exact,5.6617,28.3083,35.33,1.000,0.2744,146.48
1,IVFFlat_nprobe=4,0.0921,0.4603,2172.57,0.833,5.4959,148.76
2,IVFFlat_nprobe=8,0.1216,0.6078,1645.16,0.888,5.4959,148.76
3,IVFFlat_nprobe=16,0.1653,0.8267,1209.63,0.922,5.4959,148.76
4,IVFFlat_nprobe=32,0.2533,1.2665,789.60,0.945,5.4959,148.76
5,IVFFlat_nprobe=64,0.2902,1.4508,689.29,0.964,5.4959,148.76


## 7. HNSWFlat

In [10]:
hnsw = faiss.IndexHNSWFlat(d, HNSW_M, faiss.METRIC_INNER_PRODUCT)
hnsw.hnsw.efConstruction = HNSW_EF_CONSTRUCTION

t0 = time.perf_counter()
hnsw.add(xb)
build_sec = time.perf_counter() - t0
size_mb = get_size_mb(hnsw)

for ef in HNSW_EF_SEARCHES:
    hnsw.hnsw.efSearch = ef
    row = run_search(hnsw, f"HNSWFlat_ef={ef}", gt_ids)
    row["build_sec"] = round(build_sec, 4)
    row["size_mb"] = size_mb
    results.append(row)

pd.DataFrame(results)

,index_name,search_sec,ms_per_query,qps,recall@10,build_sec,size_mb
0,FlatIP_exact,5.6617,28.3083,35.33,1.0000,0.2744,146.48
1,IVFFlat_nprobe=4,0.0921,0.4603,2172.57,0.8330,5.4959,148.76
2,IVFFlat_nprobe=8,0.1216,0.6078,1645.16,0.8880,5.4959,148.76
3,IVFFlat_nprobe=16,0.1653,0.8267,1209.63,0.9220,5.4959,148.76
4,IVFFlat_nprobe=32,0.2533,1.2665,789.60,0.9450,5.4959,148.76
5,IVFFlat_nprobe=64,0.2902,1.4508,689.29,0.9640,5.4959,148.76
6,HNSWFlat_ef=16,0.0239,0.1196,8362.22,0.9370,21.9402,172.44
7,HNSWFlat_ef=32,0.0289,0.1446,6915.70,0.9525,21.9402,172.44
8,HNSWFlat_ef=64,0.0330,0.1648,6069.56,0.9710,21.9402,172.44
9,HNSWFlat_ef=128,0.0625,0.3124,3201.48,0.9745,21.9402,172.44


## 8. So sánh và chọn cấu hình

In [11]:
df = pd.DataFrame(results)

cols = [
    "index_name",
    "build_sec",
    "size_mb",
    "search_sec",
    "ms_per_query",
    "qps",
    f"recall@{TOPK}",
]

df = df[cols]

base_ms = df.loc[df["index_name"] == "FlatIP_exact", "ms_per_query"].iloc[0]
df["speedup_vs_flat"] = (base_ms / df["ms_per_query"]).round(2)

df

,index_name,build_sec,size_mb,search_sec,ms_per_query,qps,recall@10,speedup_vs_flat
0,FlatIP_exact,0.2744,146.48,5.6617,28.3083,35.33,1.0000,1.00
1,IVFFlat_nprobe=4,5.4959,148.76,0.0921,0.4603,2172.57,0.8330,61.50
2,IVFFlat_nprobe=8,5.4959,148.76,0.1216,0.6078,1645.16,0.8880,46.58
3,IVFFlat_nprobe=16,5.4959,148.76,0.1653,0.8267,1209.63,0.9220,34.24
4,IVFFlat_nprobe=32,5.4959,148.76,0.2533,1.2665,789.60,0.9450,22.35
5,IVFFlat_nprobe=64,5.4959,148.76,0.2902,1.4508,689.29,0.9640,19.51
6,HNSWFlat_ef=16,21.9402,172.44,0.0239,0.1196,8362.22,0.9370,236.69
7,HNSWFlat_ef=32,21.9402,172.44,0.0289,0.1446,6915.70,0.9525,195.77
8,HNSWFlat_ef=64,21.9402,172.44,0.0330,0.1648,6069.56,0.9710,171.77
9,HNSWFlat_ef=128,21.9402,172.44,0.0625,0.3124,3201.48,0.9745,90.62


In [12]:
good = df[df[f"recall@{TOPK}"] >= MIN_RECALL].copy()

if len(good) > 0:
    best = good.sort_values("ms_per_query").iloc[0]
else:
    best = df.sort_values([f"recall@{TOPK}", "ms_per_query"], ascending=[False, True]).iloc[0]

best

index_name         HNSWFlat_ef=64
build_sec                 21.9402
size_mb                    172.44
search_sec                  0.033
ms_per_query               0.1648
qps                       6069.56
recall@10                   0.971
speedup_vs_flat            171.77
Name: 8, dtype: object